<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%89%B5%E5%BB%BA%E8%B3%87%E6%96%99%E9%9B%86/selenium_crawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Colab 進行matplotlib繪圖時顯示繁體中文
# 下載台北思源黑體並命名taipei_sans_tc_beta.ttf，移至指定路徑
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib

# 改style要在改font之前
# plt.style.use('seaborn')

matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')
from google.colab import files
import pandas as pd

def load_data() -> pd.DataFrame:
  uploaded = files.upload()

  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))

  # 將上傳的檔案讀取到 pandas DataFrame 中
  # 假設上傳的檔案是 CSV 格式。如果是其他格式，您可能需要調整讀取函數（例如：pd.read_excel）
  file_name = next(iter(uploaded))
  df = pd.read_csv(file_name)

  # 顯示 DataFrame 的前 5 行
  display(df.head())
  return df
# df = load_data()
def save_df(df, filename = 'phishing_dataset_for_trainning'):
    """
    將指定的 DataFrame 存成 CSV 檔案。

    參數:
    - df: 要儲存的 DataFrame
    - filename: 存檔的檔名（例如 'output.csv'）

    功能:
    - 使用 UTF-8 with BOM 編碼避免中文亂碼
    - 不包含索引欄位
    """
    try:
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✅ 資料已成功儲存至 '{filename}'")
    except Exception as e:
        print(f"❌ 儲存失敗：{e}")

In [ ]:
%%capture
!pip install google-generativeai
!pip install selenium
!pip install webdriver-manager
!apt-get update
# !apt install chromium-chromedriver
# 2. 手動安裝與最新 WebDriver 相符的 Chrome 瀏覽器
#    這能確保版本匹配，避免啟動錯誤
# !wget -q https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/114.0.5735.90/win64/chrome-win64.zip
# !unzip -q chrome-win64.zip -d /bin/

In [ ]:
%%capture
!pip install google-colab-selenium

# **AI**

In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
# from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
import os
import numpy as np
from urllib3.exceptions import MaxRetryError, ReadTimeoutError
import gc
from google_colab_selenium import Chrome

# --- 設定 ---
FILE_NAME = "/content/phishing_dataset_expansion_1.csv"  # 你的資料集檔案名稱
NEW_FILE_NAME = '/content/phishing_dataset_expansion_1_Gemini_text.csv'
NEW_COLUMN_NAME = 'visible_text'                    # 我們要創建的新欄位
SAVE_INTERVAL = 10                                  # 每處理 N 筆資料就儲存一次，防止中斷
RENDER_WAIT_TIME = 3                                # 載入頁面後，等待 JS 渲染的秒數

def setup_driver():
    """初始化 Selenium WebDriver"""
    print("正在初始化 WebDriver...")
    options = Options()
    options.add_argument('--headless')  # 在背景執行，不開啟瀏覽器視窗
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36")
    options.page_load_strategy = 'eager'
    # --- [!!! 新增此區塊以提高安全性 !!!] ---
    # 設定瀏覽器的下載行為
    prefs = {
        # "download.default_directory": "/dev/null",  # 將下載指向一個無效位置 (Linux/macOS)
        "download.default_directory": "NUL",      # (如果是 Windows, 用這個)
        "download.prompt_for_download": False,      # 不詢問下載位置
        "download.directory_upgrade": True,
        "profile.default_content_settings.popups": 0, # 封鎖彈出視窗
        "safebrowsing.enabled": False,                # 關閉安全瀏覽 (避免它干擾爬蟲)
        "profile.default_content_setting_values.automatic_downloads": 2 # *** 關鍵：禁止自動下載 ***
    }
    options.add_experimental_option("prefs", prefs)
    # --- [安全設定結束] ---
    try:
        driver = Chrome(options=options)
        # 設定頁面載入和腳本執行的超時時間
        driver.set_page_load_timeout(30)  # 30秒內頁面必須載入
        driver.set_script_timeout(10)   # 10秒內腳本必須執行完畢
        print("WebDriver 初始化完成。")
        return driver
    except Exception as e:
        print(f"WebDriver 初始化失敗: {e}")
        print("請確保你已安裝 Google Chrome 瀏覽器。")
        return None

def fetch_visible_text(driver, url):
    """
    (爬取文本的方法)
    使用 Selenium 爬取指定 URL 的可見文本 (document.body.innerText)
    """
    if not isinstance(url, str) or not url.strip():
        return "FETCH_ERROR: Invalid URL"

    # 確保 URL 有 http/https 協議頭
    if not url.startswith('http://') and not url.startswith('https://'):
        url = 'http://' + url

    try:
        driver.get(url)
        time.sleep(1)
        page_source_lower = driver.page_source.lower()

        if "dns_probe_finished_nxdomain" in page_source_lower or "err_name_not_resolved" in page_source_lower:
            print(f"  [Info] 網站不存在 (DNS): {url}")
            return "FETCH_ERROR: DNS_PROBE_FINISHED_NXDOMAIN"
        if "err_connection_refused" in page_source_lower:
            print(f"  [Info] 連線被拒: {url}")
            return "FETCH_ERROR: ERR_CONNECTION_REFUSED"
        if "err_connection_timed_out" in page_source_lower:
            print(f"  [Info] 連線超時: {url}")
            return "FETCH_ERROR: ERR_CONNECTION_TIMED_OUT"

        # 等待固定的秒數，讓 JavaScript 有時間渲染頁面
        time.sleep(RENDER_WAIT_TIME)

        # 執行 JS 來獲取 innerText
        text = driver.execute_script("return document.body.innerText;")

        if text is None:
             return "FETCH_EMPTY: 頁面未回傳可見文本"

        # --- [!!! 這是你要求的新清潔邏輯 !!!] ---

        # 1. 將文本按 "換行" 拆分為陣列
        lines = text.split('\n')

        # 2. 遍歷每一行，去除前後空白，並只保留 "非空" 的行
        non_empty_lines = [line.strip() for line in lines if line.strip()]

        # 3. 如果過濾後沒有任何內容，回傳 EMPTY
        if not non_empty_lines:
            if "<frame" in page_source_lower:
                return "FETCH_EMPTY: 頁面為 <frame> 結構"
            return "FETCH_EMPTY: 頁面未回傳可見文本 (清潔後)"

        # 4. 將乾淨的行用 "單一空格" 串接成一個字串
        cleaned_text = '\n'.join(non_empty_lines)
        print(f'清潔後的文本: {cleaned_text}')
        return cleaned_text
        # --- [清潔邏輯結束] ---


    except TimeoutException:
        print(f"  [Error] 頁面載入超時: {url}")
        return "FETCH_ERROR: Page load timed out"
    except WebDriverException as e:
        error_msg = str(e).split('\n')[0]
        print(f"  [Error] WebDriver 錯誤: {error_msg}")
        return f"FETCH_ERROR: {error_msg}"

    except Exception as e:
        error_msg = str(e).split('\n')[0]
        print(f"  [Error] 未知錯誤: {error_msg}")
        return f"FETCH_ERROR: Unknown error - {error_msg}"

def process_dataset(df, column_name, file_to_save):
    """
    (主要處理方法)
    遍歷 DataFrame，呼叫爬蟲，並即時更新資料集
    """
    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料需要處理...")
    nb_driver = 200
    # 遍歷 DataFrame 的每一行
    driver = setup_driver()
    try:
        for index, row in df.iterrows():

            # 檢查 'visible_text' 欄位是否為空 (pd.isna) 或為空字串
            # 如果已有內容，則跳過，實現「斷點續爬」
            if pd.isna(row[column_name]) or row[column_name] == "":
                url = row['url']
                print(f"正在處理第 {index+1}/{total_rows} 筆: {url}")
                if (index+1) % nb_driver == 0:
                    driver.quit()
                    clean_memory()
                    driver = setup_driver()
                # (呼叫爬取文本的方法)
                visible_text = fetch_visible_text(driver, url)
                if isinstance(visible_text, str) and visible_text.startswith("FETCH_ERROR"):
                    if "Read timed out" in visible_text or "HTTPConnectionPool" in visible_text:
                        print("🚨 偵測到 Driver 連線逾時，正在重啟 Driver...")
                        if driver:
                            clean_memory()
                            driver.quit()
                            gc.collect()
                        driver = setup_driver()
                        print('重試一次...')
                        visible_text = fetch_visible_text(driver, url)
                # (將文本回傳後，直接更新資料集)
                # 使用 .at 來精確、快速地更新單一儲存格
                df.at[index, column_name] = visible_text

                # 每隔 N 筆資料就儲存一次檔案
                if (index + 1) % SAVE_INTERVAL == 0:
                    print(f"--- 已處理 {index+1} 筆，正在儲存進度... ---")
                    df.to_csv(file_to_save, index=False)

            else:
                # 如果該欄位已有資料，則跳過
                print(f"跳過第 {index+1}/{total_rows} 筆 (已有資料)")

        print("進度已儲存。")
        print("所有資料處理完畢。")
        driver.quit()
        clean_memory()
        return df
    except KeyboardInterrupt:
        driver.quit()
        clean_memory()
        raise KeyboardInterrupt

import os

def clean_memory():
    # 強制殺死所有 Chrome 相關進程
    os.system("pkill -9 -f chrome")
    os.system("pkill -9 -f chromedriver")
    print("🧹 已強制清理 Chrome 殘留進程與記憶體")

# --- 主程式執行 ---
if __name__ == "__main__":

    # 1. 讀入資料集
    if os.path.exists(NEW_FILE_NAME):
        # 如果新檔案已存在，表示我們上次跑到一半，從這裡繼續
        print(f"找到進度檔: {NEW_FILE_NAME}。正在載入並繼續任務...")
        try:
            df = pd.read_csv(NEW_FILE_NAME)
        except Exception as e:
            print(f"讀取 {NEW_FILE_NAME} 時發生錯誤: {e}。")
            print(f"將嘗試從原始檔案 {FILE_NAME} 重新開始。")
            try:
                df = pd.read_csv(FILE_NAME)
            except Exception as e_orig:
                 print(f"連讀取 {FILE_NAME} 都失敗: {e_orig}。程式終止。")
                 exit()
    else:
        # 如果新檔案不存在，表示這是第一次執行，從原始檔案載入
        print(f"找不到進度檔。正在從原始檔案 {FILE_NAME} 載入...")
        try:
            df = pd.read_csv(FILE_NAME)
            print(f"成功讀取資料集: {FILE_NAME}")
        except FileNotFoundError:
            print(f"錯誤: 找不到原始檔案 '{FILE_NAME}'。請確認檔案名稱與路徑。")
            exit()
        except Exception as e:
            print(f"讀取 {FILE_NAME} 時發生錯誤: {e}")
            exit()

    # 2. 若資料集不存在我們需要的欄位，就先創建欄位
    if NEW_COLUMN_NAME not in df.columns:
        print(f"找不到欄位 '{NEW_COLUMN_NAME}'，正在新增...")
        df[NEW_COLUMN_NAME] = ""  # 初始化為空字串
    else:
        print(f"找到欄位 '{NEW_COLUMN_NAME}'，將繼續處理未填滿的資料。")
        # 將可能的 NaN (Not a Number) 轉為空字串，方便後續判斷
        df[NEW_COLUMN_NAME] = df[NEW_COLUMN_NAME].fillna("")

    try:
        # 3. 傳入資料集進行處理
        clean_memory()
        df_updated= process_dataset(df, NEW_COLUMN_NAME, NEW_FILE_NAME)

        # (最後回傳新資料集) - 並儲存最終版本
        print("正在儲存最終資料集...")
        df_updated.to_csv(NEW_FILE_NAME, index=False)
        print("任務完成！")
    except KeyboardInterrupt:
        # 如果使用者手動中斷 (Ctrl+C)
        print("\n偵測到手動中斷... 正在儲存目前進度...")
        df.to_csv(NEW_FILE_NAME, index=False)
    except Exception as e:
        print(f"主程式發生錯誤: {e}")
        print("正在嘗試儲存目前進度...")
        df.to_csv(NEW_FILE_NAME, index=False)
    finally:
        # 無論如何都要關閉瀏覽器
        print("正在關閉 WebDriver...")
# [Error] 未知錯誤: HTTPConnectionPool(host='localhost', port=51371): Read timed out. (read timeout=120)

找到進度檔: /content/phishing_dataset_expansion_1_Gemini_text.csv。正在載入並繼續任務...
找到欄位 'visible_text'，將繼續處理未填滿的資料。
🧹 已強制清理 Chrome 殘留進程與記憶體
總共 50000 筆資料需要處理...
正在初始化 WebDriver...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

串流輸出內容已截斷至最後 5000 行。
跳過第 8765/50000 筆 (已有資料)
跳過第 8766/50000 筆 (已有資料)
跳過第 8767/50000 筆 (已有資料)
跳過第 8768/50000 筆 (已有資料)
跳過第 8769/50000 筆 (已有資料)
跳過第 8770/50000 筆 (已有資料)
跳過第 8771/50000 筆 (已有資料)
跳過第 8772/50000 筆 (已有資料)
跳過第 8773/50000 筆 (已有資料)
跳過第 8774/50000 筆 (已有資料)
跳過第 8775/50000 筆 (已有資料)
跳過第 8776/50000 筆 (已有資料)
跳過第 8777/50000 筆 (已有資料)
跳過第 8778/50000 筆 (已有資料)
跳過第 8779/50000 筆 (已有資料)
跳過第 8780/50000 筆 (已有資料)
跳過第 8781/50000 筆 (已有資料)
跳過第 8782/50000 筆 (已有資料)
跳過第 8783/50000 筆 (已有資料)
跳過第 8784/50000 筆 (已有資料)
跳過第 8785/50000 筆 (已有資料)
跳過第 8786/50000 筆 (已有資料)
跳過第 8787/50000 筆 (已有資料)
跳過第 8788/50000 筆 (已有資料)
跳過第 8789/50000 筆 (已有資料)
跳過第 8790/50000 筆 (已有資料)
跳過第 8791/50000 筆 (已有資料)
跳過第 8792/50000 筆 (已有資料)
跳過第 8793/50000 筆 (已有資料)
跳過第 8794/50000 筆 (已有資料)
跳過第 8795/50000 筆 (已有資料)
跳過第 8796/50000 筆 (已有資料)
跳過第 8797/50000 筆 (已有資料)
跳過第 8798/50000 筆 (已有資料)
跳過第 8799/50000 筆 (已有資料)
跳過第 8800/50000 筆 (已有資料)
跳過第 8801/50000 筆 (已有資料)
跳過第 8802/50000 筆 (已有資料)
跳過第 8803/50000 筆 (已有資料)
跳過第 8804/50000 筆 (已有資料)
跳過第 8805/50000 筆 (已

🧹 已強制清理 Chrome 殘留進程與記憶體

偵測到手動中斷... 正在儲存目前進度...
正在關閉 WebDriver...


# **對抗性樣本**

In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
# from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
import os


from google_colab_selenium import Chrome

# --- 設定 ---
FILE_NAME = "/content/phishing_dataset_expansion_forEmbeddingModule_Gemini.csv"  # 你的資料集檔案名稱
NEW_FILE_NAME = '/content/Adversarial_df_safe_http_test_ai.csv'
NEW_COLUMN_NAME = 'visible_text'                    # 我們要創建的新欄位
SAVE_INTERVAL = 10                                  # 每處理 N 筆資料就儲存一次，防止中斷
RENDER_WAIT_TIME = 3                                # 載入頁面後，等待 JS 渲染的秒數

def setup_driver():
    """初始化 Selenium WebDriver"""
    print("正在初始化 WebDriver...")
    options = Options()
    options.add_argument('--headless')  # 在背景執行，不開啟瀏覽器視窗
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36")
    # --- [!!! 新增此區塊以提高安全性 !!!] ---
    # 設定瀏覽器的下載行為
    prefs = {
        # "download.default_directory": "/dev/null",  # 將下載指向一個無效位置 (Linux/macOS)
        "download.default_directory": "NUL",      # (如果是 Windows, 用這個)
        "download.prompt_for_download": False,      # 不詢問下載位置
        "download.directory_upgrade": True,
        "profile.default_content_settings.popups": 0, # 封鎖彈出視窗
        "safebrowsing.enabled": False,                # 關閉安全瀏覽 (避免它干擾爬蟲)
        "profile.default_content_setting_values.automatic_downloads": 2 # *** 關鍵：禁止自動下載 ***
    }
    options.add_experimental_option("prefs", prefs)
    # --- [安全設定結束] ---
    try:
        driver = Chrome(options=options)
        # 設定頁面載入和腳本執行的超時時間
        driver.set_page_load_timeout(30)  # 30秒內頁面必須載入
        driver.set_script_timeout(10)   # 10秒內腳本必須執行完畢
        print("WebDriver 初始化完成。")
        return driver
    except Exception as e:
        print(f"WebDriver 初始化失敗: {e}")
        print("請確保你已安裝 Google Chrome 瀏覽器。")
        return None

def fetch_visible_text(driver, url):
    """
    (爬取文本的方法)
    使用 Selenium 爬取指定 URL 的可見文本 (document.body.innerText)
    """
    if not isinstance(url, str) or not url.strip():
        return "FETCH_ERROR: Invalid URL"

    # 確保 URL 有 http/https 協議頭
    if not url.startswith('http://') and not url.startswith('https://'):
        url = 'http://' + url

    try:
        driver.get(url)
        page_source_lower = driver.page_source.lower()

        if "dns_probe_finished_nxdomain" in page_source_lower or "err_name_not_resolved" in page_source_lower:
            print(f"  [Info] 網站不存在 (DNS): {url}")
            return "FETCH_ERROR: DNS_PROBE_FINISHED_NXDOMAIN"
        if "err_connection_refused" in page_source_lower:
            print(f"  [Info] 連線被拒: {url}")
            return "FETCH_ERROR: ERR_CONNECTION_REFUSED"
        if "err_connection_timed_out" in page_source_lower:
            print(f"  [Info] 連線超時: {url}")
            return "FETCH_ERROR: ERR_CONNECTION_TIMED_OUT"

        # 等待固定的秒數，讓 JavaScript 有時間渲染頁面
        time.sleep(RENDER_WAIT_TIME)

        # 執行 JS 來獲取 innerText
        text = driver.execute_script("return document.body.innerText;")

        if text is None:
             return "FETCH_EMPTY: 頁面未回傳可見文本"

        # --- [!!! 這是你要求的新清潔邏輯 !!!] ---

        # 1. 將文本按 "換行" 拆分為陣列
        lines = text.split('\n')

        # 2. 遍歷每一行，去除前後空白，並只保留 "非空" 的行
        non_empty_lines = [line.strip() for line in lines if line.strip()]

        # 3. 如果過濾後沒有任何內容，回傳 EMPTY
        if not non_empty_lines:
            if "<frame" in page_source_lower:
                return "FETCH_EMPTY: 頁面為 <frame> 結構"
            return "FETCH_EMPTY: 頁面未回傳可見文本 (清潔後)"

        # 4. 將乾淨的行用 "單一空格" 串接成一個字串
        cleaned_text = '\n'.join(non_empty_lines)
        print(f'清潔後的文本: {cleaned_text}')
        return cleaned_text
        # --- [清潔邏輯結束] ---


    except TimeoutException:
        print(f"  [Error] 頁面載入超時: {url}")
        return "FETCH_ERROR: Page load timed out"
    except WebDriverException as e:
        error_msg = str(e).split('\n')[0]
        print(f"  [Error] WebDriver 錯誤: {error_msg}")
        return f"FETCH_ERROR: {error_msg}"
    except Exception as e:
        error_msg = str(e).split('\n')[0]
        print(f"  [Error] 未知錯誤: {error_msg}")
        return f"FETCH_ERROR: Unknown error - {error_msg}"

def process_dataset(df, column_name, file_to_save):
    """
    (主要處理方法)
    遍歷 DataFrame，呼叫爬蟲，並即時更新資料集
    """
    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料需要處理...")
    nb_driver = 200
    # 遍歷 DataFrame 的每一行
    driver = setup_driver()
    try:
        for index, row in df.iterrows():

            # 檢查 'visible_text' 欄位是否為空 (pd.isna) 或為空字串
            # 如果已有內容，則跳過，實現「斷點續爬」
            if pd.isna(row[column_name]) or row[column_name] == "":
                url = row['url']
                print(f"正在處理第 {index+1}/{total_rows} 筆: {url}")
                if (index+1) % nb_driver == 0:
                    driver.quit()
                    driver = setup_driver()
                # (呼叫爬取文本的方法)
                visible_text = fetch_visible_text(driver, url)

                # (將文本回傳後，直接更新資料集)
                # 使用 .at 來精確、快速地更新單一儲存格
                df.at[index, column_name] = visible_text

                # 每隔 N 筆資料就儲存一次檔案
                if (index + 1) % SAVE_INTERVAL == 0:
                    print(f"--- 已處理 {index+1} 筆，正在儲存進度... ---")
                    df.to_csv(file_to_save, index=False)

            else:
                # 如果該欄位已有資料，則跳過
                print(f"跳過第 {index+1}/{total_rows} 筆 (已有資料)")

        print("進度已儲存。")
        print("所有資料處理完畢。")
        driver.quit()
        return df
    except KeyboardInterrupt:
        driver.quit()
        raise KeyboardInterrupt

# --- 主程式執行 ---
if __name__ == "__main__":

    # 1. 讀入資料集
    if os.path.exists(NEW_FILE_NAME):
        # 如果新檔案已存在，表示我們上次跑到一半，從這裡繼續
        print(f"找到進度檔: {NEW_FILE_NAME}。正在載入並繼續任務...")
        try:
            df = pd.read_csv(NEW_FILE_NAME)
        except Exception as e:
            print(f"讀取 {NEW_FILE_NAME} 時發生錯誤: {e}。")
            print(f"將嘗試從原始檔案 {FILE_NAME} 重新開始。")
            try:
                df = pd.read_csv(FILE_NAME)
            except Exception as e_orig:
                 print(f"連讀取 {FILE_NAME} 都失敗: {e_orig}。程式終止。")
                 exit()
    else:
        # 如果新檔案不存在，表示這是第一次執行，從原始檔案載入
        print(f"找不到進度檔。正在從原始檔案 {FILE_NAME} 載入...")
        try:
            df = pd.read_csv(FILE_NAME)
            print(f"成功讀取資料集: {FILE_NAME}")
        except FileNotFoundError:
            print(f"錯誤: 找不到原始檔案 '{FILE_NAME}'。請確認檔案名稱與路徑。")
            exit()
        except Exception as e:
            print(f"讀取 {FILE_NAME} 時發生錯誤: {e}")
            exit()

    # 2. 若資料集不存在我們需要的欄位，就先創建欄位
    if NEW_COLUMN_NAME not in df.columns:
        print(f"找不到欄位 '{NEW_COLUMN_NAME}'，正在新增...")
        df[NEW_COLUMN_NAME] = ""  # 初始化為空字串
    else:
        print(f"找到欄位 '{NEW_COLUMN_NAME}'，將繼續處理未填滿的資料。")
        # 將可能的 NaN (Not a Number) 轉為空字串，方便後續判斷
        df[NEW_COLUMN_NAME] = df[NEW_COLUMN_NAME].fillna("")

    try:
        # 3. 傳入資料集進行處理
        df_updated= process_dataset(df, NEW_COLUMN_NAME, NEW_FILE_NAME)

        # (最後回傳新資料集) - 並儲存最終版本
        print("正在儲存最終資料集...")
        df_updated.to_csv(NEW_FILE_NAME, index=False)
        print("任務完成！")
    except KeyboardInterrupt:
        # 如果使用者手動中斷 (Ctrl+C)
        print("\n偵測到手動中斷... 正在儲存目前進度...")
        df.to_csv(NEW_FILE_NAME, index=False)
    except Exception as e:
        print(f"主程式發生錯誤: {e}")
        print("正在嘗試儲存目前進度...")
        df.to_csv(NEW_FILE_NAME, index=False)
    finally:
        # 無論如何都要關閉瀏覽器
        print("正在關閉 WebDriver...")

找到進度檔: /content/Adversarial_df_safe_http_test_ai.csv。正在載入並繼續任務...
找不到欄位 'visible_text'，正在新增...
總共 14 筆資料需要處理...
正在初始化 WebDriver...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

WebDriver 初始化完成。
正在處理第 1/14 筆: http://kktours.com.hk/vendor/phpunit/phpunit/src/util/php/paypai/paypay/xb_ppl/signin
清潔後的文本: 404
頁面錯誤。
返回上一頁 返回首頁
正在處理第 2/14 筆: http://www.medikalhms.com/myn/vendor/phpunit/phpunit/src/util/php/def/portal/
清潔後的文本: 404 Page Not Found
The page you requested was not found.
正在處理第 3/14 筆: http://www.sorrentinovini.com/wp-content/plugins/mail-boxes-etc/lib/vendor/setasign/fpdi/src/pdfparser/filter/invoice/docusign/index2.php
清潔後的文本: ERROR: PAGE NOT FOUND
404
This page isn’t available.
Go to Homepage
正在處理第 4/14 筆: http://www.mastersenergyplastics.com/wordpress/wp-content/plugins/5128097f765a43b7935ea3e3cc4e893f/y/mm/mmd/etisalat/login.alibaba-com/
  [Error] WebDriver 錯誤: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
正在處理第 5/14 筆: http://gxnwys.com//wp-admin/dev/cncountin/cncountin/cncountin/cncountin/cncountin/upload/index.php?email=kissoons@guysuco.com
清潔後的文本: 404 Not Found
nginx
正在處理第 6/14 筆: http://qulckbooksinvolce.com/sima/index2.php/index2_files/__ma

# **playwright**

In [ ]:
%%capture
!pip install playwright
!playwright install chromium
!playwright install-deps

In [ ]:
import pandas as pd
import time
import os
import numpy as np
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
import gc

# --- 設定 ---
FILE_NAME = "/content/phishing_dataset_expansion_1.csv"
NEW_FILE_NAME = '/content/phishing_dataset_expansion_forEmbeddingModule_Gemini_text.csv'  #  phishing_dataset_expansion_2_Gemini_text
NEW_COLUMN_NAME = 'visible_text'
SAVE_INTERVAL = 10
RENDER_WAIT_TIME = 3
RESTART_INTERVAL = 300

# 注意：這裡改成 async def
from playwright.async_api import Error as PlaywrightError

async def fetch_visible_text(page, url):
    """
    使用 Playwright Async API 爬取文本 (包含 Redirect 處理)
    """
    if not isinstance(url, str) or not url.strip():
        return "FETCH_ERROR: Invalid URL"

    if not url.startswith('http://') and not url.startswith('https://'):
        url = 'http://' + url

    try:
        # --- [關鍵修改開始] ---
        # 我們將 goto 包在 try 裡面，專門捕捉 "interrupted" 錯誤
        try:
            response = await page.goto(url, timeout=30000, wait_until='domcontentloaded')
        except PlaywrightError as e:
            # 如果錯誤訊息包含 "interrupted"，表示發生了頁面跳轉
            if "interrupted by another navigation" in str(e):
                print(f"  [Info] 偵測到轉址 (Redirect)，正在等待新頁面載入: {url}")
                # 等待新頁面載入完成 (這很重要，不然會抓到空內容)
                try:
                    await page.wait_for_load_state('domcontentloaded', timeout=30000)
                except:
                    pass # 如果等待超時，就盡量抓現有的
                response = None # 轉址後 response 物件可能不準確，設為 None
            else:
                # 如果是其他錯誤 (如連線失敗)，則正常拋出
                raise e
        # --- [關鍵修改結束] ---

        # 檢查 HTTP 狀態碼 (只有在沒有發生轉址錯誤且 response 存在時才檢查)
        if response and response.status >= 400:
             return f"FETCH_ERROR: HTTP Status {response.status}"

        # 等待 JS 渲染
        await page.wait_for_timeout(RENDER_WAIT_TIME * 1000)

        # 獲取 innerText
        text = await page.inner_text("body")

        if not text:
             return "FETCH_EMPTY: 頁面未回傳可見文本"

        lines = text.split('\n')
        non_empty_lines = [line.strip() for line in lines if line.strip()]

        if not non_empty_lines:
            content = await page.content()
            if "<frame" in content.lower():
                return "FETCH_EMPTY: 頁面為 <frame> 結構"
            return "FETCH_EMPTY: 頁面未回傳可見文本 (清潔後)"

        cleaned_text = '\n'.join(non_empty_lines)
        return cleaned_text

    except PlaywrightTimeoutError:
        print(f"  [Error] 頁面載入超時: {url}")
        return "FETCH_ERROR: Page load timed out"
    except (KeyboardInterrupt, asyncio.CancelledError):
            print("\n偵測到手動中斷 (Ctrl+C)...")
            # 在 async 中我們通常讓錯誤往上拋，由外層捕捉來存檔
            raise KeyboardInterrupt
    except Exception as e:
        error_msg = str(e).split('\n')[0]
        # 這裡為了不讓版面太亂，如果還是跳轉錯誤，我們視為成功抓取 (因為下面會繼續跑)
        if "interrupted" in error_msg:
            return "FETCH_ERROR: Redirect Interrupted (Content might be missing)"

        print(f"  [Error] 錯誤: {error_msg}")
        return f"FETCH_ERROR: {error_msg}"

# 注意：這裡改成 async def
async def process_dataset(df, column_name, file_to_save):
    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料需要處理...")

    # 注意：使用 async_playwright
    async with async_playwright() as p:
        # 注意：加上 await
        browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-setuid-sandbox'])

        # 注意：加上 await
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36",
            ignore_https_errors=True,
            java_script_enabled=True
        )

        # 資源攔截函式也必須是 async
        async def block_agressive_resources(route):
            if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
                await route.abort()
            else:
                await route.continue_()

        # 注意：加上 await
        page = await context.new_page()
        # 注意：加上 await
        await page.route("**/*", block_agressive_resources)

        try:
            for index, row in df.iterrows():
                if pd.isna(row[column_name]) or row[column_name] == "":
                    url = row['url']
                    print(f"正在處理第 {index+1}/{total_rows} 筆: {url}")
                    # 注意：加上 await
                    visible_text = await fetch_visible_text(page, url)

                    # 這裡只印出前50個字，保持介面乾淨
                    print(f"爬蟲結果: {visible_text}" if visible_text else "無結果")

                    df.at[index, column_name] = visible_text

                    if (index + 1) % SAVE_INTERVAL == 0:
                        print(f"--- 已處理 {index+1} 筆，正在儲存進度... ---")
                        df.to_csv(file_to_save, index=False)

                    if (index + 1) % RESTART_INTERVAL == 0:
                        print(f"♻️ 達到 {RESTART_INTERVAL} 筆限制，強制關閉 Browser 釋放記憶體...")
                        try:
                            await page.close()
                            await context.close()
                            await browser.close()
                        except:
                            pass
                        browser = None
                        context = None
                        page = None
                        gc.collect() # 強制 Python 回收記憶體
                        browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-setuid-sandbox'])
                        context = await browser.new_context(
                            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36",
                            ignore_https_errors=True,
                            java_script_enabled=True
                        )
                        page = await context.new_page()
                        await page.route("**/*", block_agressive_resources)
                    # 重啟 Page 機制
                    elif (index + 1) % 100 == 0:
                        print("♻️ 重啟 Page 以釋放記憶體...")
                        await page.close()
                        await context.clear_cookies()
                        page = await context.new_page()
                        await page.route("**/*", block_agressive_resources)

                else:
                    pass

        except (KeyboardInterrupt, asyncio.CancelledError):
            print("\n偵測到手動中斷 (Ctrl+C)...")
            # 在 async 中我們通常讓錯誤往上拋，由外層捕捉來存檔
            print("正在儲存目前進度並退出...")
            df.to_csv(NEW_FILE_NAME, index=False)
            print("進度已保存。")
            raise KeyboardInterrupt
        except Exception as e:
            print(f"發生未預期錯誤: {e}")
        finally:
            print("關閉瀏覽器...")
            await browser.close()

    return df

# --- 主程式執行區塊 ---
# 在 Colab 中，我們不能使用 if __name__ == "__main__" 來執行 async 函式
# 我們直接定義一個 main() 並 await 它

async def main():
    # 1. 讀入資料集
    if os.path.exists(NEW_FILE_NAME):
        print(f"找到進度檔: {NEW_FILE_NAME}。正在載入並繼續任務...")
        try:
            df = pd.read_csv(NEW_FILE_NAME)
        except Exception as e:
            print(f"讀取進度檔失敗，嘗試讀取原始檔... {e}")
            df = pd.read_csv(FILE_NAME)
    else:
        print(f"找不到進度檔。正在從原始檔案 {FILE_NAME} 載入...")
        try:
            df = pd.read_csv(FILE_NAME)
        except FileNotFoundError:
            print(f"錯誤: 找不到原始檔案 '{FILE_NAME}'")
            return

    if NEW_COLUMN_NAME not in df.columns:
        df[NEW_COLUMN_NAME] = ""
    else:
        df[NEW_COLUMN_NAME] = df[NEW_COLUMN_NAME].fillna("")

    # 2. 執行處理
    try:
        os.system("pkill -9 -f chrome") # 清理舊程序

        # 注意：這裡使用 await
        df_updated = await process_dataset(df, NEW_COLUMN_NAME, NEW_FILE_NAME)

        print("正在儲存最終資料集...")
        df_updated.to_csv(NEW_FILE_NAME, index=False)
        print("任務完成！")

    except (KeyboardInterrupt, asyncio.CancelledError):
        print("進度已保存。")

# --- 啟動 Async 任務 ---
# 在 Jupyter/Colab 中，直接 await 函式即可
await main()

串流輸出內容已截斷至最後 5000 行。
Single Family
Size
2,688 SqFt
Rooms
4 Beds + 2.5 Baths
Open House
View Details Similar Properties Save
Covington
$ 445,000
Type
Single Family
Size
2,424 SqFt
Rooms
4 Beds + 3 Baths
Open House
View Details Similar Properties Save
New Orleans
$ 350,000
Type
Single Family
Size
1,950 SqFt
Rooms
4 Beds + 2.5 Baths
Open House
View Details Similar Properties Save
New Orleans
$ 598,000
Type
Condos
Size
1,855 SqFt
Rooms
3 Beds + 2.5 Baths
Open House
View Details Similar Properties Save
Metairie
$ 399,000
Type
Single Family
Size
1,892 SqFt
Rooms
3 Beds + 2 Baths
Open House
View Details Similar Properties Save
Lacombe
$ 389,000
Type
Single Family
Size
1,816 SqFt
Rooms
3 Beds + 2 Baths
Open House
View Details Similar Properties Save
New Orleans
$ 1,495,000
Type
Single Family
Size
3,295 SqFt
Rooms
4 Beds + 3.5 Baths
Open House
View Details Similar Properties Save
Mandeville
$ 570,000
Type
Single Family
Size
3,381 SqFt
Rooms
5 Beds + 3.5 Baths
Open House
View Details Similar Pro